# 00 — Data Download

Downloads posts and comments from the r/changemyview subreddit via the Reddit API and saves them as a timestamped JSON file in `data/`.

**Before running:** obtain Reddit API credentials at [reddit.com/prefs/apps](https://www.reddit.com/prefs/apps) (create a "script" app), then add them to a `.env` file in the project root:

```
REDDIT_CLIENT_ID=...
REDDIT_CLIENT_SECRET=...
REDDIT_USERNAME=...
REDDIT_PASSWORD=...
```

This notebook is independent of all other notebooks and does not need to be re-run unless you want to extend the dataset with newer posts.

In [6]:
# Based on collect_posts.py by J. Hunter Priniski & Zachary Horne
# https://github.com/jpriniski/CMV
import os
from dotenv import load_dotenv

load_dotenv()

# --- Configuration ---
SORT_MODE  = 'top'   # 'top' for all-time top posts; 'new' to extend with recent posts
LIMIT      = 500     # number of posts to collect

OUTPUT_DIR = os.path.abspath('../data')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Sort mode : {SORT_MODE}')
print(f'Limit     : {LIMIT}')
print(f'Output dir: {OUTPUT_DIR}')

In [7]:
import praw
import time
from prawcore.exceptions import TooManyRequests

reddit = praw.Reddit(
    client_id     = os.environ['REDDIT_CLIENT_ID'],
    client_secret = os.environ['REDDIT_CLIENT_SECRET'],
    username      = os.environ['REDDIT_USERNAME'],
    password      = os.environ['REDDIT_PASSWORD'],
    user_agent    = 'cmv-data-collector/1.0',
)
print('Authenticated as:', reddit.user.me())

Authenticated as: Few-Holiday-2564


In [8]:
def get_replies(comment_dict):
    replies = comment_dict['_replies']._comments
    return [r.id for r in replies] if replies else []

def get_comments_with_retry(submission, retries=3, delay=60):
    for attempt in range(retries):
        try:
            return submission.comments.replace_more(limit=50)
        except TooManyRequests:
            if attempt < retries - 1:
                print(f'Rate limit hit. Retrying in {delay}s...')
                time.sleep(delay)
            else:
                raise

COMMENT_FIELDS = (
    '_mod', 'subreddit_id', 'approved_at_utc', 'ups', 'mod_reason_by',
    'banned_by', 'removal_reason', 'link_id', 'author_flair_type',
    'author_flair_template_id', 'likes', 'no_follow', 'user_reports',
    'saved', 'id', 'banned_at_utc', 'mod_reason_title', 'gilded',
    'archived', 'report_reasons', 'author', 'can_mod_post',
    'send_replies', 'parent_id', 'score', 'approved_by',
    'downs', 'body', 'edited', 'author_flair_css_class',
    'collapsed', 'author_flair_richtext', 'is_submitter',
    'collapsed_reason', 'body_html', 'stickied', 'subreddit_type',
    'can_gild', 'author_flair_text_color', 'score_hidden',
    'permalink', 'num_reports', 'name', 'created', 'author_flair_text',
    'created_utc', 'subreddit_name_prefixed', 'controversiality',
    'depth', 'author_flair_background_color', 'mod_reports', 'mod_note',
    'distinguished', '_fetched', '_info_params',
)

REMOVED = {'[deleted]', '[removed]'}

def get_comments(submission):
    get_comments_with_retry(submission)
    com_tree = submission.comments[:]
    flat = []
    while com_tree:
        comment = com_tree.pop(0)
        com_tree.extend(comment.replies)
        flat.append(comment)

    result = []
    for comment in flat:
        d = vars(comment)
        if d.get('body') in REMOVED:
            continue
        try:
            row = {f: d[f] for f in COMMENT_FIELDS if f in d and d[f]}
        except KeyError:
            continue
        row['_replies'] = get_replies(d)
        if 'author' in row:
            row['author'] = vars(row['author'])['name']
        result.append(row)
    return result

In [9]:
POST_FIELDS = (
    'approved_at_utc', 'selftext', 'user_reports', 'saved',
    'mod_reason_title', 'gilded', 'clicked', 'title', 'link_flair_richtext',
    'subreddit_name_prefixed', 'hidden', 'pwls', 'link_flair_css_class',
    'downs', 'parent_whitelist_status', 'hide_score', 'name', 'quarantine',
    'link_flair_text_color', 'author_flair_background_color', 'subreddit_type',
    'ups', 'domain', 'media_embed', 'author_flair_template_id', 'is_original_content',
    'secure_media', 'is_reddit_media_domain', 'is_meta', 'category', 'secure_media_embed',
    'link_flair_text', 'can_mod_post', 'score', 'approved_by', 'thumbnail', 'edited',
    'author_flair_css_class', 'author_flair_richtext', 'content_categories', 'is_self',
    'mod_note', 'created', 'link_flair_type', 'wls', 'banned_by', 'author_flair_type',
    'contest_mode', 'selftext_html', 'likes', 'suggested_sort', 'banned_at_utc',
    'view_count', 'archived', 'no_follow', 'is_crosspostable', 'pinned', 'over_18',
    'media_only', 'link_flair_template_id', 'can_gild', 'spoiler', 'locked',
    'author_flair_text', 'visited', 'num_reports', 'distinguished',
    'subreddit_id', 'mod_reason_by', 'removal_reason', 'link_flair_background_color', 'id',
    'report_reasons', 'author', 'num_crossposts', 'num_comments', 'send_replies', 'mod_reports',
    'author_flair_text_color', 'permalink', 'whitelist_status', 'stickied', 'url',
    'subreddit_subscribers', 'created_utc', 'media', 'is_video',
    '_fetched', '_info_params', 'comment_limit', 'comment_sort', '_flair', '_mod',
)

import json, datetime

subreddit   = reddit.subreddit('changemyview')
feed        = subreddit.top(limit=LIMIT) if SORT_MODE == 'top' else subreddit.new(limit=LIMIT)
posts       = []

for submission in feed:
    print(f'Collecting: {submission.title[:60]}')
    d = vars(submission)
    if d.get('author') is None:
        print('  skipped — deleted post')
        continue
    try:
        row = {f: d[f] for f in POST_FIELDS if f in d and d[f]}
    except KeyError:
        continue
    row['_comments'] = get_comments(submission)
    row['author']    = vars(row['author'])['name']
    posts.append(row)

stamp    = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
out_path = os.path.join(OUTPUT_DIR, f'{stamp}_posts.json')
with open(out_path, 'w') as f:
    json.dump(posts, f)

print(f'\nSaved → {out_path}')

Collecting: CMV: Mike Bloomberg's campaign is proof that the ultra wealt
Collecting: CMV: Kanye West is a shill for president Trump and running t
Collecting: CMV: Most Americans who oppose a national healthcare system 
Collecting: CMV: Donald Trump has not made a single lasting positive imp
Collecting: CMV: being a conservative is the least Christ-like political
Collecting: cmv: The ‘Work hard and don’t give up’ message common on Red
Collecting: CMV: politicians should be required to wear NASCAR-style jum
Collecting: CMV: Democrats and Republicans live in completely different 
Collecting: CMV: "Folks" is a reasonably inclusive, gender neutral term,
  skipped — deleted post
Collecting: CMV: US presidential candidates should take an exam covering
Collecting: CMV: Antivax doctors and nurses (and other licensed healthca
Collecting: CMV: As an anti-Trump voter, all the nitpicking at President
Collecting: CMV: Off-duty cops are just civilians and should have none o
Collecting: CMV: Will Smit

KeyboardInterrupt: 

In [ ]:
total_comments = sum(len(p.get('_comments', [])) for p in posts)
print(f'Posts collected   : {len(posts)}')
print(f'Comments collected: {total_comments}')